# Day 14 — PCA와 heatmap: 샘플 전체를 한눈에 보기

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/week-2/day-14.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/week-2/day-14.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 많은 유전자가 달라진 경우: PCA가 그룹을 가른다

유전자 300개 중 60개가 FLT에서 2배 높은 가상 실험입니다.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

def simulate(n_de, seed, n_genes=300, effect=2.0):
    """샘플 12개(GC 6, FLT 6) × 유전자 n_genes개의 가상 raw count를 만든다."""
    rng = np.random.default_rng(seed)
    group = np.array(["GC"] * 6 + ["FLT"] * 6)
    samples = [f"GC{i}" for i in range(1, 7)] + [f"FLT{i}" for i in range(1, 7)]
    level = rng.lognormal(4, 1, n_genes)
    fold = np.ones((n_genes, 12))
    fold[:n_de, 6:] = effect                           # 앞의 n_de개 유전자만 FLT에서 2배
    depth = rng.uniform(0.7, 1.4, 12)
    mu = level[:, None] * fold * depth[None, :]
    counts = rng.negative_binomial(n=20, p=20 / (20 + mu))
    counts = pd.DataFrame(counts, index=[f"gene{i:03d}" for i in range(n_genes)], columns=samples)
    return counts, pd.Series(group, index=samples, name="group")

def log_cpm(counts):
    return np.log2(counts / counts.sum() * 1_000_000 + 1)

def run_pca(counts):
    x = log_cpm(counts).T                              # 행 = 샘플, 열 = 유전자
    x = (x - x.mean()) / x.std()                       # 유전자별 표준화(z-score)
    pca = PCA(n_components=2)
    pcs = pd.DataFrame(pca.fit_transform(x), index=x.index, columns=["PC1", "PC2"])
    return pcs, pca.explained_variance_ratio_

counts, group = simulate(n_de=60, seed=14)
pcs, ratio = run_pca(counts)
print("설명 분산 비율:", ratio.round(2))
print(pcs.join(group).groupby("group")[["PC1", "PC2"]].mean().round(1))

### 예시 2 — 몇 개의 유전자만 달라진 경우: PCA는 못 보지만 차이는 있다

이번에는 300개 중 5개만 FLT에서 2배 높습니다.

In [ ]:
counts2, group2 = simulate(n_de=5, seed=17)            # 300개 중 5개만 진짜로 다르다
pcs2, ratio2 = run_pca(counts2)
print("설명 분산 비율:", ratio2.round(2))
print("PC1 순서대로 샘플 나열:", list(pcs2["PC1"].sort_values().index))
print("PC2 순서대로 샘플 나열:", list(pcs2["PC2"].sort_values().index))

one_gene = log_cpm(counts2).loc["gene000"]             # 진짜로 다른 5개 중 하나
print("\ngene000의 샘플별 log2(CPM+1):")
print(one_gene.round(1).to_dict())

### 예시 3 — 품질이 나쁜 샘플 하나가 PCA를 차지한다

예시 1의 데이터에서 FLT6 샘플 하나만 유전자마다 무작위로 값을 흔들어 망가뜨립니다.

In [ ]:
counts3, group3 = simulate(n_de=60, seed=14)
noise = np.random.default_rng(0).lognormal(0, 1.0, len(counts3))
counts3["FLT6"] = (counts3["FLT6"] * noise).round().astype(int)   # FLT6만 품질이 나쁜 샘플로 만든다
pcs3, ratio3 = run_pca(counts3)
print("설명 분산 비율:", ratio3.round(2))
print(pcs3.round(1))

## 직접 해보기

1. 어제 만든 `notebooks/02_osdr_qc_and_pca.ipynb`에 `## Day 14 — PCA와 heatmap 연습 (가상 데이터)` 섹션을 추가하고 예시 1을 실행합니다.
2. PCA 그림을 그립니다. 축 라벨에 설명 분산 비율을 넣고, 점마다 샘플 이름을 붙입니다.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

d = pcs.join(group)
fig, ax = plt.subplots(figsize=(5, 4))
sns.scatterplot(data=d, x="PC1", y="PC2", hue="group", s=90, ax=ax)
for name, row in d.iterrows():
    ax.annotate(name, (row["PC1"], row["PC2"]), fontsize=8, xytext=(4, 4), textcoords="offset points")
ax.set_xlabel(f"PC1 ({ratio[0]:.0%})")
ax.set_ylabel(f"PC2 ({ratio[1]:.0%})")
plt.tight_layout()
plt.savefig("../figures/day14_pca.png", dpi=150)
plt.show()

3. heatmap을 그립니다. 가장 많이 변하는 유전자 30개를 골라, 유전자별 z-score로 색칠하고, 행과 열을 군집화합니다. 열 위의 색 띠는 그룹을 나타냅니다(회색 GC, 주황 FLT).

In [ ]:
lc = log_cpm(counts)
top = lc.var(axis=1).sort_values(ascending=False).head(30).index     # 가장 많이 변하는 유전자 30개
g = sns.clustermap(lc.loc[top], z_score=0, cmap="vlag", center=0, figsize=(7, 8),
                   col_colors=group.map({"GC": "gray", "FLT": "orange"}))
g.savefig("../figures/day14_heatmap.png", dpi=150)
plt.show()

이 레슨을 만들 때 실행한 결과에서는 열이 FLT 6개와 GC 6개로 깔끔하게 두 가지로 나뉘었고, 고른 30개 유전자 중 14개가 우리가 실제로 차이를 넣은 유전자(gene000–gene059)였습니다. 나머지 16개는 무엇 때문에 "많이 변하는 유전자"로 뽑혔을지 생각해 보세요(힌트: 발현이 낮은 유전자의 잡음).
4. 예시 2와 예시 3을 실행합니다. 각각 2단계의 그림 코드에서 `pcs`, `group`, `ratio`를 `pcs2`, `group2`, `ratio2`(또는 `pcs3`, `group3`, `ratio3`)로 바꿔 PCA 그림을 그려 봅니다(파일 이름도 `day14_pca_ex2.png`처럼 바꿉니다). 세 그림을 나란히 놓고 어떤 점이 다른지 봅니다.
5. 예시 2 데이터에서 gene000을 개별 샘플 그림으로 그립니다.

In [ ]:
one = pd.DataFrame({"group": group2, "log2(CPM+1)": log_cpm(counts2).loc["gene000"]})
fig, ax = plt.subplots(figsize=(3.5, 4))
sns.stripplot(data=one, x="group", y="log2(CPM+1)", size=8, color="black", ax=ax)
ax.set_title("gene000")
plt.tight_layout()
plt.show()

## 🤖 AI와 함께 하기 (주간 실습, 20–30분)

3. **AI의 4번 예측을 시뮬레이션으로 확인합니다.** AI가 "발현량이 큰 유전자가 PC1을 지배한다"고 답했다면, 아래 코드로 직접 봅니다. 발현량은 크지만 군 차이가 없는 유전자 하나와, 작지만 군 차이가 있는 유전자 두 개를 만들었습니다.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
n = 12                                        # 샘플 12개 (가상)
group = np.array([0]*6 + [1]*6)               # 앞 6개 = A군, 뒤 6개 = B군

big = rng.normal(10000, 2000, n)              # 발현량은 크지만 군 차이 없는 유전자
small1 = 5 + 2*group + rng.normal(0, 0.5, n)  # 작지만 군 차이 있는 유전자 1
small2 = 8 + 2*group + rng.normal(0, 0.5, n)  # 작지만 군 차이 있는 유전자 2
X = np.column_stack([big, small1, small2])

def pc1_loading(M):
    M = M - M.mean(axis=0)                    # 열(유전자)마다 평균을 뺀다
    _, _, vt = np.linalg.svd(M, full_matrices=False)
    return np.round(np.abs(vt[0]), 2)         # PC1이 각 유전자에 주는 가중치

print("스케일링 전 PC1 가중치 [big, small1, small2]:", pc1_loading(X))
print("스케일링 후 PC1 가중치 [big, small1, small2]:", pc1_loading(X / X.std(axis=0)))